# Gold Layer – Data Quality Checks

These queries look for suspicious values and missing dimension attributes in the Gold fact and denormalised analytics tables. A returned row is a candidate for investigation; it is not automatically an error without considering the business rules.

In [ ]:
-- 1. Non-positive quantities
SELECT *
FROM retail_gold_db.fact_order_items
WHERE quantity <= 0;

In [ ]:
-- 2. Negative prices or revenue
SELECT *
FROM retail_gold_db.fact_order_items
WHERE unit_price < 0
   OR net_amount < 0
   OR net_amount_inr < 0;

In [ ]:
-- 3. Missing product attributes after denormalisation
SELECT COUNT(*) AS records_with_missing_product_attributes
FROM retail_gold_db.gld_business_analytics
WHERE sku IS NULL
   OR category_name IS NULL
   OR brand_name IS NULL;

In [ ]:
-- 4. Missing date attributes after denormalisation
SELECT COUNT(*) AS records_with_missing_date_attributes
FROM retail_gold_db.gld_business_analytics
WHERE year IS NULL
   OR month_name IS NULL
   OR day_name IS NULL;

In [ ]:
-- 5. Compare fact rows with denormalised rows
SELECT
    (SELECT COUNT(*) FROM retail_gold_db.fact_order_items) AS fact_rows,
    (SELECT COUNT(*) FROM retail_gold_db.gld_business_analytics) AS analytics_rows;

## Important interpretation

The denormalised table uses INNER JOINs to date and product dimensions. If `analytics_rows` is lower than `fact_rows`, some fact records did not find a matching date or product key. If it is higher, investigate duplicate keys in the joined dimensions, especially `dim_products.product_id`.